# Notebook 3 — Data Quality Assessment
### Sprint 4 | Data Inspection & Exploratory Data Analysis (EDA)

**Structure for every concept below:** Concept → Example → Implementation → Output
→ Interpretation → Insight → AI/ML Relevance.

**Dataset:** Telco Customer Churn (7,043 customers, 21 columns) — documented in
`01_Dataset_Understanding.ipynb`. Notebook 2 already flagged one suspicious finding:
`.info()` reports **zero** missing values, yet `TotalCharges` is stored as text. This
notebook is where that gets properly investigated, alongside duplicates and other invalid
values — the **Identify Problems** step of this sprint's guiding principle.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("telco_churn.csv")
print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")


Dataset loaded: 7,043 rows, 21 columns


---
# Part A — Missing Values


---
## A1. Number of Missing Values

### Concept
The number of missing values is a simple count, per column, of cells that are empty
(`NaN`/`None`) according to Pandas' own detection — the starting point of any missing-data
investigation.

### Example
**Business example:** A telecom billing system might fail to record `TotalCharges` for a
customer who joined and cancelled within the same billing cycle before their first
invoice was generated.

**Why it matters during data analysis:** I can't decide how to *handle* missing data
before knowing exactly *how much* exists, and where.

### Implementation


In [2]:
missing_counts = df.isnull().sum()
print(missing_counts)


customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


### Output
Every single column reports **0** missing values.

### Interpretation
Taken at face value, this says the dataset is perfectly complete — not one cell is
officially missing anywhere.

### Insight
This is misleading, and I already have a specific reason to distrust it: `TotalCharges` is
stored as a text dtype (`object` or `str`, depending on the Pandas version) rather than
a numeric type, and text columns can hide "missing"
data as blank strings (`""`) or whitespace instead of a true `NaN` — which `.isnull()`
would not catch. This needs to be checked directly, not assumed away by `.isnull()` alone.

### AI/ML Relevance
Trusting `.isnull().sum()` alone here would cause a churn model to silently receive
corrupted (non-numeric, blank) values in `TotalCharges` instead of being properly
imputed — a realistic, easy-to-miss production bug.


---
## A2. Percentage of Missing Values

### Concept
Expressing missing values as a percentage of total rows makes severity comparable across
columns of different sizes — "11 missing" means something different in a 100-row dataset
than in a 7,043-row one.

### Example
**Business example:** A column missing 2% of its values is usually safe to impute; a
column missing 70% of its values may need to be dropped entirely rather than imputed.

### Implementation


In [3]:
# First, properly detect the HIDDEN missing values in TotalCharges (blank/whitespace strings)
total_charges_numeric = pd.to_numeric(df['TotalCharges'], errors='coerce')
hidden_missing_count = total_charges_numeric.isnull().sum()

print(f"TotalCharges — missing values found by .isnull() alone : {df['TotalCharges'].isnull().sum()}")
print(f"TotalCharges — missing values found after proper numeric conversion: {hidden_missing_count}")
print(f"TotalCharges — percentage missing: {hidden_missing_count / len(df) * 100:.2f}%")


TotalCharges — missing values found by .isnull() alone : 0
TotalCharges — missing values found after proper numeric conversion: 11
TotalCharges — percentage missing: 0.16%


### Output
`.isnull()` alone finds 0 missing values in `TotalCharges`. Converting the column to
numeric with `errors='coerce'` (which turns anything that can't be parsed as a number
into `NaN`) reveals **11** hidden missing values — about **0.16%** of all rows.

### Interpretation
The 11 blank-string entries were invisible to `.isnull()` because Pandas was treating
them as valid (if unusual) text, not as missing data — `pd.to_numeric(..., errors='coerce')`
is what actually exposes them.

### Insight
0.16% missing is a genuinely small proportion — small enough that dropping, or more
sensibly imputing, these 11 rows would have minimal impact on the overall dataset.

### AI/ML Relevance
This is a textbook example of why raw `.isnull()` checks are not sufficient on their own —
a proper data-quality pass must also check whether numeric-looking columns are actually
stored as numeric, and coerce-and-recheck when they aren't.


---
## A3. Missing Values by Column

### Concept
Breaking missing-value counts down by column (rather than one dataset-wide total) shows
exactly *where* to focus cleaning effort — different columns often need entirely
different handling strategies.

### Example
**AI/ML use case:** A feature-engineering pipeline typically applies a different
imputation strategy per column (median for skewed numerics, mode for categoricals) — so a
per-column breakdown is a direct input to that decision.

### Implementation


In [4]:
df_corrected = df.copy()
df_corrected['TotalCharges'] = pd.to_numeric(df_corrected['TotalCharges'], errors='coerce')

missing_by_column = df_corrected.isnull().sum()
missing_pct_by_column = (missing_by_column / len(df_corrected) * 100).round(2)

missing_summary = pd.DataFrame({
    'missing_count': missing_by_column,
    'missing_pct': missing_pct_by_column
})
missing_summary = missing_summary[missing_summary['missing_count'] > 0]
print(missing_summary)


              missing_count  missing_pct
TotalCharges             11         0.16


### Output
After correcting `TotalCharges`'s data type, exactly one column shows missing values:
`TotalCharges`, with 11 missing (0.16%). Every other column remains genuinely complete.

### Interpretation
This dataset's missingness problem is small, isolated to a single column, and now fully
quantified — not a widespread issue across many columns.

### Insight
Because only one column is affected, and only slightly, this is a low-risk missing-data
situation — a simple, well-justified imputation (or row removal, given how few rows are
affected) will be a reasonable fix in the next sprint.

### AI/ML Relevance
A per-column missing-value table like this is typically the very first artifact handed
from an EDA phase to a data-cleaning phase in a real ML project.


---
## A4. Missing Value Patterns

### Concept
Missing value *patterns* look at whether missingness is random, or whether it correlates
with other variables (e.g., missing `TotalCharges` only happening for a specific type of
customer) — this distinction matters because it affects which imputation strategy is
statistically appropriate.

### Example
**Business example:** If missing `TotalCharges` only happens for brand-new customers, that
missingness isn't random — it has a clear, explainable cause connected to `tenure`.

### Implementation


In [5]:
rows_with_missing_total_charges = df_corrected[df_corrected['TotalCharges'].isnull()]
print(f"Number of rows with missing TotalCharges: {len(rows_with_missing_total_charges)}")
print("\ntenure values for these specific rows:")
print(rows_with_missing_total_charges['tenure'].value_counts())


Number of rows with missing TotalCharges: 11

tenure values for these specific rows:
tenure
0    11
Name: count, dtype: int64


### Output
All 11 rows with a missing `TotalCharges` have `tenure == 0`.

### Interpretation
This is not random missingness — it has a clear, logical explanation: these are brand-new
customers (0 months of tenure) who haven't been billed yet, so there's genuinely no
"total charges" figure to record for them.

### Insight
This changes the right fix entirely: rather than imputing with a mean/median (which would
invent a plausible-looking but meaningless number for a customer who hasn't been billed),
the logically correct value for these 11 rows is **0** — they haven't paid anything yet.

### AI/ML Relevance
Understanding *why* data is missing (not just *that* it's missing) is exactly what
separates a thoughtful imputation strategy from a blindly applied one — this is a direct,
concrete example of the "investigate why before you fill" principle.


---
# Part B — Duplicate Records


---
## B1. Identify Duplicates

### Concept
(Recap from Sprint 3) `.duplicated()` flags rows that are exact repeats of an earlier row
— the direct tool for spotting accidental double-entry of the same record.

### Example
**Business example:** If the same customer's signup was accidentally submitted twice
through a web form, their full record might appear twice in this dataset.

### Implementation


In [6]:
duplicate_flags = df.duplicated()
print(f"Any duplicate rows found: {duplicate_flags.any()}")
print(f"Number of duplicate rows: {duplicate_flags.sum()}")


Any duplicate rows found: False
Number of duplicate rows: 0


### Output
`0` duplicate rows found across the entire dataset.

### Interpretation
No row is an exact, full duplicate of any other row.

### Insight
This is a clean result — but it only checks for *fully identical* rows across all 21
columns, which is a strict definition of "duplicate" (checked further in B4).

### AI/ML Relevance
Undetected exact duplicates would let the same customer's information influence a model
more than once, subtly biasing what the model learns.


---
## B2. Count Duplicates

### Concept
Beyond just detecting *whether* duplicates exist, counting them precisely quantifies how
much of the dataset is affected.

### Example
**AI/ML use case:** If 5% of rows were exact duplicates, that would meaningfully inflate
some patterns in the data — knowing the exact count tells me how much correction is
needed.

### Implementation


In [7]:
print(f"Total rows: {len(df):,}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Percentage of dataset that is duplicated: {df.duplicated().sum() / len(df) * 100:.2f}%")


Total rows: 7,043
Duplicate rows: 0
Percentage of dataset that is duplicated: 0.00%


### Output
0 duplicates out of 7,043 rows — 0.00%.

### Interpretation
There is no exact-duplicate problem in this dataset at all.

### Insight
This particular data-quality risk can be marked as "checked, not present" for this
dataset — worth documenting explicitly in the final report (Notebook 13) rather than
silently omitting it, since a reviewer would reasonably ask whether this was checked.

### AI/ML Relevance
Documenting "checked, and clean" is just as valuable as documenting "found a problem" —
it shows the check was actually performed, not skipped.


---
## B3. Identify Completely Duplicated Rows

### Concept
"Completely duplicated" specifically means every single column matches another row
exactly — the strictest possible duplicate definition, as opposed to a looser
same-customer-different-details definition (covered next).

### Example
**Business example:** Two rows with the same `customerID` AND identical values in every
other column would be a completely duplicated row — a pure data-entry accident.

### Implementation


In [8]:
fully_duplicated_rows = df[df.duplicated(keep=False)]
print(f"Completely duplicated rows found: {len(fully_duplicated_rows)}")
if len(fully_duplicated_rows) > 0:
    print(fully_duplicated_rows)
else:
    print("None found — confirmed clean on this strict definition.")


Completely duplicated rows found: 0
None found — confirmed clean on this strict definition.


### Output
0 rows — confirmed clean, consistent with B1 and B2.

### Interpretation
This dataset has no completely-duplicated rows under any definition of "exact match."

### Insight
Given `customerID` is already confirmed unique (Notebook 1), this result was expected —
if the identifier is unique, a fully duplicated row (which would need a duplicated
`customerID` too) is structurally impossible here.

### AI/ML Relevance
This cross-check (unique IDs implying no full duplicates) is a good sanity habit —
results from different checks should logically agree with each other.


---
## B4. Identify Potential Duplicate Records

### Concept
A "potential" duplicate is a looser, more realistic definition — rows that match on the
*meaningful business fields* (excluding the identifier) even if they don't match on
every single column. This can catch the same person accidentally entered as two different
"customers."

### Example
**Business example:** The same person, re-entered as a "new" customer under a different
`customerID` after a billing system glitch, would share identical demographic and service
details but have a different ID — invisible to a strict full-row duplicate check.

### Implementation


In [9]:
business_columns = [c for c in df.columns if c != 'customerID']
potential_duplicates = df[df.duplicated(subset=business_columns, keep=False)]

print(f"Rows sharing identical values on every column EXCEPT customerID: {len(potential_duplicates)}")
if len(potential_duplicates) > 0:
    print(potential_duplicates.sort_values(business_columns).head(10))


Rows sharing identical values on every column EXCEPT customerID: 42
      customerID  gender  SeniorCitizen Partner Dependents  tenure  \
6491  9728-FTTVZ  Female              0      No         No       1   
6764  7660-HDPJV  Female              0      No         No       1   
4495  4702-IOQDC  Female              0      No         No       1   
6267  0328-GRPMV  Female              0      No         No       1   
5522  2619-WFQWU  Female              0      No         No       1   
5759  9985-MWVIX  Female              0      No         No       1   
542   2866-IKBTM  Female              0      No         No       1   
1491  8605-ITULD  Female              0      No         No       1   
5170  7721-DVEKZ  Female              0      No         No       1   
6774  0970-QXPXW  Female              0      No         No       1   

     PhoneService MultipleLines InternetService       OnlineSecurity  ...  \
6491          Yes            No     Fiber optic                   No  ...   
6764   

### Output
42 rows are found — pairs (or small groups) of customers who share identical values
across every business column, differing only in `customerID`.

### Interpretation
This does NOT necessarily mean these are true duplicate people — with 16 categorical
columns that each only have 2-4 possible values, it's statistically plausible for two
genuinely different customers to coincidentally share the exact same profile (e.g., two
different female customers, no partner, no dependents, 1 month of tenure, same contract
type, same services).

### Insight
This is exactly the kind of finding that needs human judgment, not an automatic
"remove them" rule — the sprint brief's outlier-handling principle ("do not automatically
remove") applies just as much here. These 42 rows (about 0.6% of the dataset) should be
flagged for manual review rather than deleted outright — notice they also skew toward
very short `tenure` values (1 month), which makes coincidental profile matches even more
plausible for newer customers who haven't yet accumulated a unique combination of charges.

### AI/ML Relevance
Blindly deleting "potential duplicates" like this could remove genuine, valid customers
purely because the categorical feature space is small enough for coincidental matches —
a good example of why automated rules need a human sanity check before being applied.


---
# Part C — Invalid Values


---
## C1. Negative Values Where Inappropriate

### Concept
Some columns are logically impossible to be negative — tenure, charges, counts — so any
negative value there signals a data-entry or processing error, not a real measurement.

### Example
**Business example:** A `tenure` of -3 months would be logically impossible — time spent
as a customer cannot be negative.

### Implementation


In [10]:
numeric_cols_should_be_nonnegative = ['tenure', 'MonthlyCharges']
for col in numeric_cols_should_be_nonnegative:
    negative_count = (df[col] < 0).sum()
    print(f"{col}: {negative_count} negative values")

negative_total_charges = (total_charges_numeric < 0).sum()
print(f"TotalCharges: {negative_total_charges} negative values")


tenure: 0 negative values
MonthlyCharges: 0 negative values
TotalCharges: 0 negative values


### Output
0 negative values in `tenure`, `MonthlyCharges`, or `TotalCharges`.

### Interpretation
Every value in these three inherently non-negative columns is logically valid.

### Insight
This is a clean result on this specific check — worth noting for the final report as a
check that was performed and passed, not skipped.

### AI/ML Relevance
An unnoticed negative value in a column like this would likely indicate a serious upstream
data-pipeline bug — confirming their absence is a basic but essential trust check before
using these features.


---
## C2. Impossible Values

### Concept
Impossible values are values that fall outside any logically valid range for that column,
even if they're technically the "right" data type (e.g., a percentage of 150%, or a
`SeniorCitizen` flag that isn't 0 or 1).

### Example
**Business example:** `SeniorCitizen` should only ever be 0 or 1 (No/Yes, already encoded
as a binary flag) — any other value would be impossible given how the column is defined.

### Implementation


In [11]:
unique_senior_values = df['SeniorCitizen'].unique()
print(f"Unique values in SeniorCitizen: {unique_senior_values}")
print(f"Are all values valid (0 or 1)? {set(unique_senior_values).issubset({0, 1})}")

print(f"\nMonthlyCharges range: {df['MonthlyCharges'].min()} to {df['MonthlyCharges'].max()}")
print("(No fixed 'impossible' upper bound is known for pricing, but this range looks plausible for a telecom plan.)")


Unique values in SeniorCitizen: [0 1]
Are all values valid (0 or 1)? True

MonthlyCharges range: 18.25 to 118.75
(No fixed 'impossible' upper bound is known for pricing, but this range looks plausible for a telecom plan.)


### Output
`SeniorCitizen` contains only `[0, 1]` — fully valid. `MonthlyCharges` ranges from about
$18 to $119, a plausible range for telecom billing.

### Interpretation
No impossible values were found in either checked column.

### Insight
Binary flag columns like `SeniorCitizen` are cheap and fast to validate this way — worth
doing for every 0/1-style column as a standard check.

### AI/ML Relevance
An unnoticed value like `SeniorCitizen = 2` would silently break the assumption that this
is a clean binary feature, potentially confusing a model or an encoding step downstream.


---
## C3. Incorrect Categories

### Concept
Incorrect categories means checking whether a categorical column's actual values match its
*documented, expected* set of categories — catching typos, inconsistent capitalization, or
stray unexpected labels.

### Example
**Business example:** If `Contract` is expected to only ever be "Month-to-month," "One
year," or "Two year," but a row shows "month-to-month" (lowercase) or "1 year" instead,
that's an incorrect/inconsistent category needing standardization (Sprint 3, Notebook 12).

### Implementation


In [12]:
expected_categories = {
    'Contract': ['Month-to-month', 'One year', 'Two year'],
    'InternetService': ['DSL', 'Fiber optic', 'No'],
    'PaymentMethod': ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'],
    'Churn': ['Yes', 'No'],
}

for col, expected in expected_categories.items():
    actual = set(df[col].unique())
    unexpected = actual - set(expected)
    print(f"{col}: {'OK — no unexpected categories' if not unexpected else f'UNEXPECTED VALUES FOUND: {unexpected}'}")


Contract: OK — no unexpected categories
InternetService: OK — no unexpected categories
PaymentMethod: OK — no unexpected categories
Churn: OK — no unexpected categories


### Output
All 4 checked columns show "OK — no unexpected categories."

### Interpretation
Every category in these columns exactly matches the expected, documented set — no typos,
inconsistent casing, or stray labels.

### Insight
This dataset's categorical columns are clean and well-standardized — a good sign for how
straightforward one-hot encoding will be in the preprocessing sprint.

### AI/ML Relevance
Unchecked category inconsistencies (e.g., "Yes" and "yes" being treated as different
categories) would silently fragment a single true category into multiple encoded columns,
diluting that feature's signal.


---
## C4. Invalid Dates

### Concept
Invalid dates are date/time values that don't parse correctly, or fall outside a sensible
range (e.g., a signup date in the future).

### Example
**Business example:** A customer `signup_date` recorded as the year 2099 would be an
obviously invalid date.

### Implementation


In [13]:
date_columns = df.select_dtypes(include=['datetime64']).columns.tolist()
print(f"Date/time columns found in this dataset: {date_columns}")
print("\nAs already established in Notebook 2, this dataset contains NO date/time columns —")
print("every record is a single snapshot, not a dated event. This check is therefore N/A here.")


Date/time columns found in this dataset: []

As already established in Notebook 2, this dataset contains NO date/time columns —
every record is a single snapshot, not a dated event. This check is therefore N/A here.


### Output
An empty list — confirming (again) there are no date columns in this dataset.

### Interpretation
This specific check simply doesn't apply to this dataset, and that's worth explicitly
documenting rather than silently skipping — a reviewer should see this was considered, not
overlooked.

### Insight
None to report for this dataset, beyond confirming the absence of date columns already
noted in Notebook 2.

### AI/ML Relevance
In a dataset that DID have dates, invalid or future-dated values would need exactly this
kind of check before any date-based feature engineering (Sprint 3, Notebook 11) could be
trusted.


---
## C5. Incorrect Data Types

### Concept
(Recap from Notebook 2) An incorrect data type means a column's logical nature (numeric,
categorical, boolean) doesn't match how Pandas is currently storing it — the central
finding of this whole notebook.

### Example
**Business example:** `TotalCharges` should be a `float`, representing money, but is
stored as text rather than numeric — the exact issue already identified.

### Implementation


In [14]:
print("Column dtype vs. logical type check:")
print(f"  TotalCharges  -> stored as: {df['TotalCharges'].dtype}, should be: float64  <- MISMATCH")
print(f"  SeniorCitizen -> stored as: {df['SeniorCitizen'].dtype}, arguably should be: bool or category (currently int64, acceptable)")
print(f"  tenure        -> stored as: {df['tenure'].dtype}, correct as: int64")
print(f"  MonthlyCharges-> stored as: {df['MonthlyCharges'].dtype}, correct as: float64")


Column dtype vs. logical type check:
  TotalCharges  -> stored as: str, should be: float64  <- MISMATCH
  SeniorCitizen -> stored as: int64, arguably should be: bool or category (currently int64, acceptable)
  tenure        -> stored as: int64, correct as: int64
  MonthlyCharges-> stored as: float64, correct as: float64


### Output
One clear mismatch confirmed: `TotalCharges` (a text dtype instead of `float64`).
`SeniorCitizen` is a minor, debatable case — it's already numeric and usable, though
arguably more semantically correct as a `category` or `bool`, given it only ever holds
0/1.

### Interpretation
This formally confirms, with full documentation, the single data-type correction this
dataset needs before it's fully clean.

### Insight
Bringing together everything checked in this notebook, `TotalCharges` needs exactly two
fixes: convert to numeric, and fill its 11 missing values with 0 (per the pattern found in
A4) — a clear, well-justified, two-step correction plan to carry into Sprint 5.

### AI/ML Relevance
This is precisely the kind of "documented problem with a documented reasoning" this
sprint asks for — not fixing it yet, but knowing exactly what needs fixing and why.


---
## C6. Unexpected Values

### Concept
Unexpected values are anything that doesn't fit the *general* shape of a column, beyond
just checking category lists — a catch-all check for surprises.

### Example
**AI/ML use case:** Scanning every categorical column's `.unique()` output, all at once,
as a final broad sweep for anything odd that more targeted checks (like C3) might have
missed.

### Implementation


In [15]:
categorical_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
categorical_cols.remove('customerID')

for col in categorical_cols:
    print(f"{col}: {df[col].unique()}")


gender: <ArrowStringArray>
['Female', 'Male']
Length: 2, dtype: str
Partner: <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
Dependents: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
PhoneService: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
MultipleLines: <ArrowStringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str
InternetService: <ArrowStringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
OnlineSecurity: <ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
OnlineBackup: <ArrowStringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str
DeviceProtection: <ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
TechSupport: <ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
StreamingTV: <ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
StreamingMovies: <ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: 

### Output
Every categorical column shows a small, clean, sensible set of values — no stray blanks,
no odd symbols, no inconsistent casing anywhere in this broad sweep.

### Interpretation
This corroborates C3's targeted check with a full, unrestricted view across every
categorical column, not just the 4 spot-checked earlier.

### Insight
The categorical side of this dataset is thoroughly confirmed clean — the only remaining
concern from this whole notebook is `TotalCharges`.

### AI/ML Relevance
A full sweep like this, even after targeted checks already passed, is good practice
before declaring a dataset "clean" — targeted checks only catch what you already thought
to look for.


---
## C7. Empty Strings

### Concept
Empty strings (`""`) are a specific, sneaky form of missing data that looks like valid
text to Pandas but carries no actual information — exactly what's hiding inside
`TotalCharges`.

### Example
**Business example:** A billing system might write an empty string instead of leaving a
field truly `NULL` when generating an export file — technically "present," practically
meaningless.

### Implementation


In [16]:
empty_string_counts = {}
for col in categorical_cols + ['TotalCharges']:
    empty_count = (df[col].astype(str).str.strip() == '').sum()
    if empty_count > 0:
        empty_string_counts[col] = empty_count

print("Columns with empty/whitespace-only strings:")
for col, count in empty_string_counts.items():
    print(f"  {col}: {count}")


Columns with empty/whitespace-only strings:
  TotalCharges: 11


### Output
Exactly one column is affected: `TotalCharges`, with 11 empty/whitespace-only strings —
matching the count already found in Part A.

### Interpretation
This directly confirms, using a completely different detection method (string-stripping
instead of numeric coercion), the same 11 problem rows found earlier — strong triangulated
evidence this is real, not a fluke of one particular detection technique.

### Insight
Having now confirmed this issue three separate ways (A2's numeric coercion, A4's pattern
analysis, and this direct string check), there's no ambiguity left about this specific
data-quality problem.

### AI/ML Relevance
Checking for empty strings explicitly (not just relying on `.isnull()`) should be a
standard part of any text-column audit, since this exact failure mode — blank string
instead of true null — is common in real exported data.


---
## C8. Null-like Values

### Concept
Null-like values are text placeholders that *represent* missing data without being an
actual empty string or true `NaN` — things like `"NA"`, `"N/A"`, `"None"`, `"null"`,
`"?"`, or `"-"`, often introduced when different systems export missing data differently.

### Example
**Business example:** A customer record merged from two different regional systems might
have "N/A" in one export and a true blank in another, for the exact same underlying
"missing" situation.

### Implementation


In [17]:
null_like_tokens = ['na', 'n/a', 'none', 'null', '?', '-', 'nan', 'unknown']

found_any = False
for col in categorical_cols:
    matches = df[col].astype(str).str.strip().str.lower().isin(null_like_tokens)
    if matches.sum() > 0:
        found_any = True
        print(f"{col}: {matches.sum()} null-like placeholder(s) found -> {df.loc[matches, col].unique()}")

if not found_any:
    print("No null-like placeholder text (e.g., 'N/A', 'None', '?') found in any categorical column.")


No null-like placeholder text (e.g., 'N/A', 'None', '?') found in any categorical column.


### Output
No null-like placeholder text found in any categorical column.

### Interpretation
This dataset's missingness problem is fully limited to the blank-string issue in
`TotalCharges` — there's no additional hidden-missing-data problem lurking behind
disguised placeholder text elsewhere.

### Insight
This is a reassuring, clean result to close out the data-quality investigation — the
dataset's *only* real, confirmed problem across this entire notebook is the
`TotalCharges` type/missingness issue.

### AI/ML Relevance
Checking for null-like tokens is especially important with datasets combined from
multiple source systems, where different upstream conventions for "missing" can easily
slip through undetected without this explicit check.


---
## Summary — Data Quality Findings

| Check | Result |
|---|---|
| Missing values (`.isnull()`) | 0 reported — **misleading** |
| Hidden missing values (after type correction) | 11, all in `TotalCharges` (0.16%) |
| Missing value pattern | All 11 have `tenure == 0` (brand-new, unbilled customers) — logical, not random |
| Exact duplicate rows | 0 |
| Potential duplicates (same profile, different ID) | 42 rows (~0.6%) — flagged for manual review, NOT auto-removed |
| Negative values in charge/tenure columns | 0 |
| Impossible values (e.g., SeniorCitizen outside 0/1) | 0 |
| Incorrect/unexpected categories | 0 |
| Invalid dates | N/A — no date columns in this dataset |
| Incorrect data types | 1 — `TotalCharges` stored as text, should be numeric |
| Empty strings | 11, in `TotalCharges` (confirms the missing-value finding) |
| Null-like placeholder text | 0 |

**Documented recommendation for Sprint 5 (Data Cleaning):**
1. Convert `TotalCharges` to numeric using `pd.to_numeric(..., errors='coerce')`.
2. Fill the resulting 11 missing values with `0`, since they all correspond to brand-new
   customers with `tenure == 0` who genuinely haven't been billed yet — not a case for
   mean/median imputation.
3. Manually review the 42 "potential duplicate" rows before deciding whether any should
   be treated specially — do not delete them automatically.

**Next notebook:** `04_Univariate_Analysis.ipynb` — analyzing numerical and categorical
variables one at a time, using the now-understood, now-documented dataset.
